# Video 6: Memory and Knowledge in CrewAI

## What We Will Build
In this notebook, we will build a **CrewAI system with memory and knowledge**. You will learn:
- How crew memory works (short-term, long-term, entity memory)
- How to enable memory in a crew
- How agents reuse previous information
- How to create and use knowledge sources
- How to test conversation context across tasks
- Comparing memory-enabled vs memory-disabled workflows

By the end, you will have a crew that remembers context across multiple interactions.

## 1. Install & Import

In [ ]:
!pip install -q crewai crewai-tools langchain-openai python-dotenv

## 2. Imports & Setup

In [ ]:
from crewai import Agent, Task, Crew, Processfrom langchain_openai import ChatOpenAIimport osfrom dotenv import load_dotenvload_dotenv()llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7, api_key=os.getenv('OPENAI_API_KEY'))print('Setup complete')

## 3. Theory: Memory in CrewAI

### Types of Memory

| Memory Type | Description | Scope |
|-------------|-------------|-------|
| **Short-term** | Context within the current execution | Single crew run |
| **Long-term** | Information persisted across runs | Multiple crew runs |
| **Entity** | Memory about specific entities (people, places, concepts) | Cross-session |

### Why Memory Matters
- Agents remember what was discussed in previous tasks
- Follow-up questions are answered with context
- Entities are tracked across the workflow
- Reduces redundant work and improves coherence

### Enabling Memory
Simply set `memory=True` when creating a Crew:
```python
Crew(
    agents=[...],
    tasks=[...],
    memory=True,  # <-- Enables all memory types
    ...
)
```

## 4. Create Agents for Memory Demo

In [ ]:
# Create agentsresearcher = Agent(    role='Context-Aware Researcher',    goal='Research topics while maintaining full context of the conversation',    backstory=(        'You are a research specialist who pays close attention to context. '        'You remember details from previous tasks and build upon them. '        'You never ask for information that was already provided.'    ),    llm=llm,    verbose=True,    allow_delegation=False)writer = Agent(    role='Contextual Writer',    goal='Write content that references and builds upon previous information',    backstory=(        'You are a writer who excels at maintaining narrative continuity. '        'You reference previous points and ensure consistency across all your writing.'    ),    llm=llm,    verbose=True,    allow_delegation=False)print('Agents created for memory demonstration')

## 5. Build a Crew WITHOUT Memory (Baseline)

In [ ]:
# Tasks that depend on context continuitytask1_no_mem = Task(    description=(        'Introduce a fictional tech startup called TechNova. Describe its founding story, '        'mission, and first product. Keep it to 200 words.'    ),    expected_output='A brief introduction to TechNova startup.',    agent=researcher)task2_no_mem = Task(    description=(        'Write a follow-up about TechNova. What challenges does it face in its second year? '        'Reference specific details from the founding story.'    ),    expected_output='A continuation of TechNova story with references to previous details.',    agent=writer,    context=[task1_no_mem])# Crew WITHOUT memorycrew_no_memory = Crew(    agents=[researcher, writer],    tasks=[task1_no_mem, task2_no_mem],    process=Process.sequential,    verbose=True,    memory=False)print('Crew WITHOUT memory created (baseline)')

## 6. Execute Without Memory

In [ ]:
print('\n' + '=' * 60)print('EXECUTING WITHOUT MEMORY')print('=' * 60)result_no_mem = crew_no_memory.kickoff()print('\n' + '=' * 60)print('OUTPUT (No Memory)')print('=' * 60)print(result_no_mem)

## 7. Build a Crew WITH Memory

In [ ]:
# Same tasks but with memory enabledtask1_mem = Task(    description=(        'Introduce a fictional tech startup called TechNova. Describe its founding story, '        'mission, and first product. Keep it to 200 words.'    ),    expected_output='A brief introduction to TechNova startup.',    agent=researcher)task2_mem = Task(    description=(        'Write a follow-up about TechNova. What challenges does it face in its second year? '        'Reference specific details from the founding story.'    ),    expected_output='A continuation of TechNova story with references to previous details.',    agent=writer,    context=[task1_mem])# Crew WITH memory enabledcrew_with_memory = Crew(    agents=[researcher, writer],    tasks=[task1_mem, task2_mem],    process=Process.sequential,    verbose=True,    memory=True  # <-- MEMORY ENABLED)print('Crew WITH memory created')

## 8. Execute With Memory

In [ ]:
print('\n' + '=' * 60)print('EXECUTING WITH MEMORY')print('=' * 60)result_with_mem = crew_with_memory.kickoff()print('\n' + '=' * 60)print('OUTPUT (With Memory)')print('=' * 60)print(result_with_mem)

## 9. Compare Memory-Enabled vs Disabled

| Aspect | No Memory | With Memory |
|--------|-----------|-------------|
| Context retention | Task-level only | Cross-task and cross-session |
| Entity tracking | None | Automatic |
| Follow-up quality | May repeat info | Builds on previous context |
| API cost | Lower | Slightly higher |
| Coherence | Variable | Higher |

### When to Use Memory
- Long-running conversations
- Multi-session workflows
- Entity-heavy tasks (names, places, concepts)
- Complex projects with many interdependent tasks

## 10. Knowledge Sources

CrewAI also supports external knowledge sources. You can provide documents that agents can reference throughout the workflow.

In [ ]:
# Create a knowledge documentknowledge_doc = """TechNova Company Profile=========================Founded: 2023Founders: Sarah Chen (CEO) and Marcus Rodriguez (CTO)Headquarters: Austin, TexasMission: To democratize AI tools for small businessesProducts:1. NovaAI - AI-powered business assistant2. NovaAnalytics - Data analysis platform3. NovaChat - Customer service chatbotFunding: $12M Series A (2024)Team Size: 45 employees"""with open('technova_knowledge.txt', 'w') as f:    f.write(knowledge_doc)print('Knowledge source created: technova_knowledge.txt')

## 11. Use Knowledge in Tasks

In [ ]:
from crewai.tools import tool@tooldef read_knowledge_document() -> str:    """Read the TechNova knowledge document."""    with open('technova_knowledge.txt', 'r') as f:        return f.read()# Agent with knowledge accessknowledge_agent = Agent(    role='Company Analyst',    goal='Analyze company information from knowledge sources',    backstory='An analyst who thoroughly reviews all available documentation.',    llm=llm,    tools=[read_knowledge_document],    verbose=True,    allow_delegation=False)knowledge_task = Task(    description=(        'Read the TechNova knowledge document and provide a comprehensive analysis. '        'Include: company overview, product analysis, funding assessment, and growth potential.'    ),    expected_output='A detailed analysis of TechNova based on the knowledge document.',    agent=knowledge_agent)knowledge_crew = Crew(    agents=[knowledge_agent],    tasks=[knowledge_task],    process=Process.sequential,    verbose=True)print('Knowledge-based crew ready')

## 12. Execute Knowledge-Based Crew

In [ ]:
print('\n' + '=' * 60)print('EXECUTING KNOWLEDGE-BASED CREW')print('=' * 60)knowledge_result = knowledge_crew.kickoff()print('\n' + '=' * 60)print('KNOWLEDGE CREW OUTPUT')print('=' * 60)print(knowledge_result)

---

## ✅ Summary

You have successfully:
1. ✅ Created a crew without memory (baseline)
2. ✅ Created a crew with memory enabled
3. ✅ Compared outputs between memory-enabled and disabled workflows
4. ✅ Created a knowledge source document
5. ✅ Built a knowledge-based agent with document access
6. ✅ Observed how memory improves context continuity

In the next notebook, we will explore **CrewAI with RAG**.